# Evaluate NSQIP model probability predictions

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
from src.data_utils import get_models, get_data
import pandas as pd

## Set up

In [ ]:
blank_file = pd.read_excel(
    BASE_PATH / "data" / "nsqip_data" / "blank_tongue_nsqip_2024.xlsx", index_col=0
)
filled_file = pd.read_excel(
    BASE_PATH / "data" / "nsqip_data" / "NSQIP_probs_2024.xlsx", index_col=0
)

Ensure no missing in filled probs

In [ ]:
# Checks length + CASEID uniqueness
assert (
    len(blank_file)
    == len(filled_file)
    == blank_file["CASEID"].nunique()
    == filled_file["CASEID"].nunique()
)
# Checks CASEIDs match
blank_ids = set(blank_file["CASEID"].unique())
filled_ids = set(filled_file["CASEID"].unique())
assert blank_ids == filled_ids

Subset

In [ ]:
rename_dict = {
    "CASEID": "CASEID",
    "Serious_Comp_No Adjustment": "serious",
    "Return_OR_No Adjustment": "reoper",
    "Surgical Site Infection_No Adjustment": "ssi",
    "Pneumonia_No Adjustment": "pneumo",
    "Any_Comp_No Adjustment": "any",
}
filled_file_sub = filled_file[list(rename_dict.keys())]
filled_file_sub = filled_file_sub.rename(columns=rename_dict).reset_index(drop=True)
filled_file_sub

Export

In [ ]:
export_path = BASE_PATH / "data" / "nsqip_data" / "nsqip_proba_processed.parquet"
if export_path.exists():
    export_path.unlink()
export_path.parent.mkdir(exist_ok=True, parents=True)
filled_file_sub.to_parquet(export_path)

# Build file of all probs

In [ ]:
# Use calibrated models
mdoel_imp_dir = BASE_PATH / "models" / "calibrated"
MODEL_LIST = [
    "svc",
    "lr",
    "nn",
    "lgbm",
    "xgb",
    "stack",
]
outcome_imp_dir = BASE_PATH / "data"
OUTCOME_LIST = [
    "ssi",
    "reoper",
    "any",
    "serious",
    "pneumo",
    # BLEED not available on NSQIP site
]
nsqip_df_full = pd.read_parquet(
    BASE_PATH / "data" / "nsqip_data" / "nsqip_proba_processed.parquet"
)

In [ ]:
export_dir = BASE_PATH / "data" / "nsqip_data" / "processed_probs"
export_dir.mkdir(exist_ok=True, parents=True)
for outcome in OUTCOME_LIST:
    print(f"{outcome}...")
    # Load NSQIP probs
    nsqip_data = nsqip_df_full[["CASEID", outcome]]
    nsqip_data[outcome] = nsqip_data[outcome] / 100
    nsqip_data = nsqip_data.rename(columns={outcome: "y_proba_nsqip"})
    # Load our data
    our_outcome_data = get_data(
        outcome_folder=f"outcome_{outcome}",
        file_dir=outcome_imp_dir / "processed_reduced",
    )
    X_test, y_test = our_outcome_data["X_test"], our_outcome_data["y_test"]
    # Load data from un-processed just for CASEID (ordering is consistent)
    y_test["CASEID"] = get_data(
        outcome_folder=f"outcome_{outcome}", file_dir=outcome_imp_dir / "processed"
    )["test_ids"]
    y_test = y_test.rename(columns={f"outcome_{outcome}": f"y_true"})
    # Join & maintain ordering of original y_test
    combined_df = y_test.merge(nsqip_data, on="CASEID", how="left").set_index("CASEID")
    ## Get our probas
    # NOTE: Raw order of these probas will be = to raw order of y_test, which was preserved
    model_dict = get_models(
        model_prefix_list=MODEL_LIST, outcome=outcome, file_dir=mdoel_imp_dir
    )
    for model_name, model in model_dict.items():
        print(f"\t{model_name}")
        cur_y_proba = model.predict_proba(X_test)[:, 1]
        combined_df[f"y_proba_{model_name}"] = cur_y_proba
    ## Export
    export_path = export_dir / f"{outcome}.parquet"
    if export_path.exists():
        export_path.unlink()
    combined_df.to_parquet(export_path)